In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import scienceplots

from pathlib import Path
from matplotlib.ticker import MultipleLocator


plt.style.use(["science", "ieee"])

tick_font = 10.4
axis_label_font = 11.0
legend_font = 8.7
panel_font = 11.0

mpl.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Times"],
    "font.size": tick_font,
    "axes.labelsize": axis_label_font,
    "xtick.labelsize": tick_font,
    "ytick.labelsize": tick_font,
    "legend.fontsize": legend_font,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "text.color": "black",
    "axes.labelcolor": "black",
    "axes.edgecolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
    "axes.grid": False,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

temperature_colors = [
    "#000000",
    "#D62728",
    "#1F77B4",
    "#2CA02C",
    "#9467BD",
    "#FF7F0E",
    "#8C564B",
]

# NICKEL MODEL
deactivation_constants_Ni = {
    "kd": np.array([18.39, 147.0]),
    "kd_c": np.array([309.87, -26.0]),
    "kd_ch4": np.array([-0.004431483, -5.376]),
    "kd_h2": np.array([-2.44388e-05, -80.19]),
}

def deactivation_constant_nickel(T_C, da_constants):
    R = 8.314462618e-3
    T_K = T_C + 273.15

    def arrhenius(parameters):
        A, E = parameters
        return A * np.exp(-E / (R * T_K))

    kd = arrhenius(da_constants["kd"])
    kd_c = arrhenius(da_constants["kd_c"])
    kd_ch4 = arrhenius(da_constants["kd_ch4"])
    kd_h2 = arrhenius(da_constants["kd_h2"])

    return kd, kd_c, kd_ch4, kd_h2


def deactivation_nickel(T_C, Pch4, Ph2, t_s, da_constants):
    kd, kd_c, kd_ch4, kd_h2 = deactivation_constant_nickel(
        T_C,
        da_constants
    )

    t_s = np.asarray(t_s, dtype=float)

    deactivation_term = (
        kd_c
        + kd_ch4 * Pch4
        + kd_h2 * Ph2**0.83
    )

    B = 0.5 * kd * deactivation_term
    denominator = 1.0 - B * t_s

    activity = np.zeros_like(
        denominator,
        dtype=float
    )

    valid = denominator > 0.0

    activity[valid] = (
        denominator[valid]**0.8
    )

    activity = np.clip(
        activity,
        0.0,
        1.0
    )

    return (
        activity.item()
        if activity.ndim == 0
        else activity
    )


temperatures_Ni = np.arange(
    550,
    701,
    25
)

#Common Pressures
Pch4_Ni = 0.5e5
Ph2_Ni = 0.0e5


# IRON MODEL
reaction_constants_Fe = {
    "k": np.array([1.283e-08, 160.0, 1073.0])
}

deactivation_constants_Fe = {
    "kd": np.array([2.0, 77.0, 1073.0])
}


def reaction_adsorption_constant_iron(
    T_C,
    r_constants
):

    T_K = T_C + 273.15
    k_ref, E, T_ref = r_constants["k"]

    R_kJ = 8.3144598e-3

    k = k_ref * np.exp(
        -(E / R_kJ)
        * (
            1.0 / T_K
            - 1.0 / T_ref
        )
    )

    delta_G0 = (
        89658.88
        - 102.27 * T_K
        - 0.00428 * T_K**2
        - 2499358.99 / T_K
    )

    R_J = 8.3144598

    Keq = np.exp(
        -delta_G0
        / (R_J * T_K)
    )

    return k, Keq


def initial_reaction_rate_iron(
    T_C,
    Pch4,
    Ph2,
    r_constants
):

    k, Keq = reaction_adsorption_constant_iron(
        T_C,
        r_constants
    )

    P0 = 100000 # 1 bar in Pa

    driving_force = (
        Pch4
        - Ph2**2
        / (Keq * P0)
    )

    return k * driving_force


def deactivation_constant_iron(
    T_C,
    da_constants
):

    T_K = T_C + 273.15

    kd_ref, Ed, T_ref = (
        da_constants["kd"]
    )

    R_kJ = 8.3144598e-3

    kd = kd_ref * np.exp(
        -(Ed / R_kJ)
        * (
            1.0 / T_K
            - 1.0 / T_ref
        )
    )

    return kd


def deactivation_iron(
    T_C,
    Pch4,
    Ph2,
    t_s,
    r_constants,
    da_constants
):

    kd = deactivation_constant_iron(
        T_C,
        da_constants
    )

    r0 = initial_reaction_rate_iron(
        T_C,
        Pch4,
        Ph2,
        r_constants
    )

    a_inf = 0.001
    M_C = 12.011

    t_s = np.asarray(
        t_s,
        dtype=float
    )

    coefficient = (
        kd
        * M_C
        * r0
    )

    denominator = (
        1.0
        - (1.0 - a_inf)
        * np.exp(
            -a_inf
            * coefficient
            * t_s
        )
    )

    activity = (
        a_inf
        / denominator
    )

    activity = np.clip(
        activity,
        a_inf,
        1.0
    )

    return (
        activity.item()
        if activity.ndim == 0
        else activity
    )


temperatures_Fe = np.arange(
    700,
    851,
    25
)

#Common Pressures
Pch4_Fe = 0.5e5
Ph2_Fe = 0.0e5


# ACTIVATED CARBON MODEL
deactivation_constants_AC = {
    "kd": np.array([943786.6667, 194.0])
}


def deactivation_constant_carbon(
    T_C,
    da_constants
):

    R = 8.314462618e-3
    T_K = T_C + 273.15

    Ad, Ed = da_constants["kd"]

    kd = Ad * np.exp(
        -Ed / (R * T_K)
    )

    return kd


def deactivation_carbon(
    T_C,
    Pch4_atm,
    t_s,
    da_constants
):

    kd = deactivation_constant_carbon(
        T_C,
        da_constants
    )

    t_s = np.asarray(
        t_s,
        dtype=float
    )

    d = 0.5
    T_K = T_C + 273.15
    T_c = 190.4

    m = 112.0 * (
        T_K / T_c
    )**(-2.9551)

    K = (
        (1.0 - d)
        * kd
        * Pch4_atm**m
    )

    denominator = (
        1.0
        - K * t_s
    )

    activity = np.zeros_like(
        denominator,
        dtype=float
    )

    valid = denominator > 0.0

    activity[valid] = (
        denominator[valid]
        ** (
            1.0
            / (1.0 - d)
        )
    )

    activity = np.clip(
        activity,
        0.0,
        1.0
    )

    return (
        activity.item()
        if activity.ndim == 0
        else activity
    )


temperatures_AC = np.arange(
    800,
    951,
    25
)

Pch4_AC_atm = (
    0.5 / 1.01325
)


#Common time range
time_h = np.linspace(
    0.0,
    2.0,
    2000
)

time_s = (
    time_h
    * 3600.0
)

#PLOT
single_fig_width = 5.15
single_fig_height = 2.474

single_left = 0.175
single_right = 0.985
single_bottom = 0.225
single_top = 0.960

def style_axis(ax):

    ax.set_xlim(
        0.0,
        2.0
    )

    ax.set_ylim(
        0.0,
        1.13
    )

    ax.set_yticks(
        np.arange(
            0.0,
            1.01,
            0.2
        )
    )

    ax.xaxis.set_major_locator(
        MultipleLocator(0.25)
    )

    ax.xaxis.set_minor_locator(
        MultipleLocator(0.05)
    )

    ax.yaxis.set_major_locator(
        MultipleLocator(0.2)
    )

    ax.yaxis.set_minor_locator(
        MultipleLocator(0.05)
    )

    ax.tick_params(
        axis="x",
        which="major",
        direction="in",
        top=True,
        bottom=True,
        labelsize=tick_font,
        colors="black",
        length=4.8,
        width=0.9,
        pad=6
    )

    ax.tick_params(
        axis="y",
        which="major",
        direction="in",
        left=True,
        right=True,
        labelsize=tick_font,
        colors="black",
        length=4.8,
        width=0.9,
        pad=6
    )

    ax.tick_params(
        axis="x",
        which="minor",
        direction="in",
        top=True,
        bottom=True,
        colors="black",
        length=2.9,
        width=0.7
    )

    ax.tick_params(
        axis="y",
        which="minor",
        direction="in",
        left=True,
        right=True,
        colors="black",
        length=2.9,
        width=0.7
    )

    ax.grid(False)

    for spine in ax.spines.values():
        spine.set_color("black")
        spine.set_linewidth(0.9)


def add_legend(ax):

    legend = ax.legend(
        ncol=4,
        loc="upper right",
        bbox_to_anchor=(0.995, 0.985),
        frameon=True,
        fancybox=False,
        framealpha=1.0,
        edgecolor="black",
        facecolor="white",
        columnspacing=0.8,
        handlelength=1.1,
        handletextpad=0.4,
        borderpad=0.25,
        labelspacing=0.14,
        fontsize=legend_font
    )

    legend.get_frame().set_linewidth(
        0.8
    )

#Nickel Figure
fig_Ni, ax_Ni = plt.subplots(
    figsize=(
        single_fig_width,
        single_fig_height
    )
)

fig_Ni.patch.set_facecolor(
    "white"
)

for T_C, color in zip(
    temperatures_Ni,
    temperature_colors
):

    activity = deactivation_nickel(
        T_C,
        Pch4_Ni,
        Ph2_Ni,
        time_s,
        deactivation_constants_Ni
    )

    ax_Ni.plot(
        time_h,
        activity,
        color=color,
        linestyle="-",
        linewidth=1.65,
        label=fr"{T_C:.0f} $^\circ\mathrm{{C}}$"
    )

ax_Ni.set_ylabel(
    r"Deactivation factor, $a$ / [-]",
    fontsize=axis_label_font,
    labelpad=8
)

ax_Ni.set_xlabel(
    r"Time / [h]",
    fontsize=axis_label_font,
    labelpad=7
)

style_axis(ax_Ni)
add_legend(ax_Ni)

fig_Ni.subplots_adjust(
    left=single_left,
    right=single_right,
    bottom=single_bottom,
    top=single_top
)

fig_Ni.savefig("deactivation_factor_Ni.pdf",
    format="pdf"
)

#Iron Figure
fig_Fe, ax_Fe = plt.subplots(
    figsize=(
        single_fig_width,
        single_fig_height
    )
)

fig_Fe.patch.set_facecolor(
    "white"
)

for T_C, color in zip(
    temperatures_Fe,
    temperature_colors
):

    activity = deactivation_iron(
        T_C,
        Pch4_Fe,
        Ph2_Fe,
        time_s,
        reaction_constants_Fe,
        deactivation_constants_Fe
    )

    ax_Fe.plot(
        time_h,
        activity,
        color=color,
        linestyle="-",
        linewidth=1.65,
        label=fr"{T_C:.0f} $^\circ\mathrm{{C}}$"
    )

ax_Fe.set_ylabel(
    r"Deactivation factor, $a$ / [-]",
    fontsize=axis_label_font,
    labelpad=8
)

ax_Fe.set_xlabel(
    r"Time / [h]",
    fontsize=axis_label_font,
    labelpad=7
)

style_axis(ax_Fe)
add_legend(ax_Fe)

fig_Fe.subplots_adjust(
    left=single_left,
    right=single_right,
    bottom=single_bottom,
    top=single_top
)

fig_Fe.savefig("deactivation_factor_Fe.pdf",
    format="pdf"
)

#Activated Carbon Figure
fig_AC, ax_AC = plt.subplots(
    figsize=(
        single_fig_width,
        single_fig_height
    )
)

fig_AC.patch.set_facecolor(
    "white"
)

for T_C, color in zip(
    temperatures_AC,
    temperature_colors
):

    activity = deactivation_carbon(
        T_C,
        Pch4_AC_atm,
        time_s,
        deactivation_constants_AC
    )

    ax_AC.plot(
        time_h,
        activity,
        color=color,
        linestyle="-",
        linewidth=1.65,
        label=fr"{T_C:.0f} $^\circ\mathrm{{C}}$"
    )

ax_AC.set_ylabel(
    r"Deactivation factor, $a$ / [-]",
    fontsize=axis_label_font,
    labelpad=8
)

ax_AC.set_xlabel(
    r"Time / [h]",
    fontsize=axis_label_font,
    labelpad=7
)

style_axis(ax_AC)
add_legend(ax_AC)

fig_AC.subplots_adjust(
    left=single_left,
    right=single_right,
    bottom=single_bottom,
    top=single_top
)

fig_AC.savefig("deactivation_factor_AC.pdf",
    format="pdf"
)

plt.show()